# [V2] PUMANet - Dual-Decoder Panoptic Segmentation

**Version 2**: PUMANet with dual-decoder architecture for joint tissue + nuclei segmentation.

**Key improvements over V1:**
- Dual-decoder: Tissue Decoder + Nuclei Decoder (NP/HV/NC heads)
- TGNA (Tissue-Guided Nuclei Attention) for cross-task fusion
- Instance segmentation via HV gradient maps (Hover-Net inspired)
- Tissue-guided post-processing for nuclei reclassification
- Mamba SSM bottleneck for global context

**Auto-resume**: Checkpoints saved to Google Drive every epoch.  
If Colab disconnects, just **Run All** again - training resumes automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset on Drive
%ls /content/drive/MyDrive/dataset_PUMA/

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/Segment_PUMA"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Segment_PUMA.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
# Install PyTorch + dependencies
!pip install -qqq torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -qqq -r {REPO_DIR}/requirements.txt
!pip install -qqq causal-conv1d mamba-ssm

In [ ]:
# Login W&B to monitor training
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
# (Optional) Keep-alive to prevent idle disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
function KeepAlive() {
    console.log("Keeping alive " + new Date().toLocaleString());
    google.colab.kernel.invokeFunction("notebook.ping", [], {});
}
setInterval(KeepAlive, 5 * 60 * 1000);
'''))
output.register_callback('notebook.ping', lambda: print('.', end='', flush=True))
print("Keep-alive enabled (ping every 5 min)")

## Train PUMANet V2

**Modes:**
- `--mode joint` : Train tissue + nuclei simultaneously (recommended)
- `--mode tissue` : Train tissue segmentation only
- `--mode nuclei` : Train nuclei segmentation only

**Tracks:**
- `--nuclei-track 1` : 4 nuclei classes (Tumor, TILs, Other)
- `--nuclei-track 2` : 11 nuclei classes (all cell types)

In [ ]:
# Joint training - Track 1 (4 nuclei classes)
%cd /content/drive/MyDrive/Segment_PUMA
!python scripts/train_v2.py \
    --devices 0 \
    --config configs/puma_v2_colab.yaml \
    --mode joint \
    --nuclei-track 1

In [ ]:
# (Optional) Joint training - Track 2 (11 nuclei classes)
# Uncomment to run Track 2 instead of / after Track 1

# %cd /content/drive/MyDrive/Segment_PUMA
# !python scripts/train_v2.py \
#     --devices 0 \
#     --config configs/puma_v2_colab.yaml \
#     --mode joint \
#     --nuclei-track 2

## Post-Processing & Evaluation

Load best checkpoint, run inference, apply tissue-guided post-processing,
and compute metrics.

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/Segment_PUMA')

import torch
import numpy as np
import matplotlib.pyplot as plt
from configs.config import load_config
from models.puma_net import PUMANet
from datasets.puma_dataset_v2 import PUMAJointDataset, puma_collate_fn
from models.postprocessing import full_pipeline
from configs.constants import get_task_config, IMAGENET_MEAN, IMAGENET_STD

# Load config & model
config = load_config('configs/puma_v2_colab.yaml')
config.NUCLEI_TRACK = 1  # Change to 2 for Track 2
config.resolve_v2()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = PUMANet.from_config(config).to(device)
ckpt = torch.load('checkpoints_v2/best_puma_v2.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {ckpt['epoch']}, score={ckpt['score']:.4f}")

In [ ]:
# Run inference + post-processing on test set
test_dataset = PUMAJointDataset(
    dataset_root=config.DATA_ROOT, split='test',
    nuclei_track=config.NUCLEI_TRACK, mode='joint',
    image_size=config.IMAGE_SIZE, use_augmentation=False,
    split_ratio=config.SPLIT_RATIO,
    use_stain_norm=getattr(config, 'USE_STAIN_NORM', True),
)

nuclei_names = get_task_config('nuclei', config.NUCLEI_TRACK).class_names
mean = np.array(IMAGENET_MEAN)
std = np.array(IMAGENET_STD)

num_vis = min(5, len(test_dataset))

for i in range(num_vis):
    sample = test_dataset[i]
    img_tensor = sample['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)

    # Extract predictions
    tissue_pred = outputs['tissue_pred'][0].argmax(0).cpu().numpy()
    np_pred = torch.softmax(outputs['np_pred'][0], dim=0)[1].cpu().numpy()
    hv_pred = outputs['hv_pred'][0].cpu().numpy()
    nc_pred = outputs['nc_pred'][0].argmax(0).cpu().numpy()

    # Post-processing: instance separation + tissue-guided refinement
    inst_map, inst_classes, centers = full_pipeline(
        np_pred, hv_pred, nc_pred, tissue_pred, nuclei_names,
    )

    # Visualize
    img_np = sample['image'].numpy()
    img_vis = (img_np * std[:, None, None] + mean[:, None, None])
    img_vis = np.clip(img_vis.transpose(1, 2, 0), 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img_vis)
    axes[0].set_title('Input')
    axes[1].imshow(tissue_pred, cmap='tab10', vmin=0, vmax=5)
    axes[1].set_title('Tissue Segmentation')
    axes[2].imshow(nc_pred, cmap='tab10', vmin=0, vmax=config.NUM_NUCLEI_CLASSES - 1)
    axes[2].set_title('Nuclei Classification')
    axes[3].imshow(inst_map, cmap='nipy_spectral')
    axes[3].set_title(f'Instances ({len(centers)} nuclei)')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"Sample {i}: {len(centers)} nuclei detected")

## Training complete

In [ ]:
# Disconnect runtime to stop billing
from google.colab import runtime
print("Training & evaluation finished! Disconnecting runtime...")
runtime.unassign()